# Support Tiers for Molecular System Forms

This document defines the support contract for molecular-system forms in the `1.x` line of MolSysMT.

Its purpose is not to list every implemented adapter. Its purpose is to make explicit:

- which forms are part of the contractual support surface;
- what kind of guarantee each tier provides;
- which forms are backed by contract verification;
- which forms are parity-verified;
- which forms are relevant to the future heavy-trajectory roadmap;
- and which areas remain outside the `1.0.0` support contract.

## How to read this document

This document separates four ideas that should not be conflated.

### 1. Support tier

The support tier tells users how strongly MolSysMT stands behind a form in the `1.x` line.

### 2. Contract verification

Contract verification means that the form is exercised by tests that validate the expected observable contract for its supported scope.

This is stronger than "the adapter exists", but different from full cross-form parity.

### 3. Parity verification

Parity verification means that equivalent molecular content represented in different supported forms is explicitly tested for equivalent results where such equivalence is part of the supported contract.

This is stronger than ordinary form support.

### 4. Heavy-mode status

Heavy-mode status indicates whether the form participates in the committed pre-`1.0.0` chunked-execution contract, is only a candidate for it, or is outside that scope.

Heavy-mode readiness must not be inferred from ordinary form support.

## Tier definitions

### Tier 1 — Contractual Forms

Tier 1 forms are part of the supported `1.x` contract.

For Tier 1 forms:

- regressions are patch-priority;
- supported semantics are expected to remain stable across the `1.x` line except for documented bug fixes and explicit support-contract revisions;
- contract support is based on implemented tests and documented scope, not only on adapter presence.

Tier 1 does not mean that every conceivable capability is guaranteed. It means that the documented supported scope of the form is part of the contractual product surface.

### Tier 2 — Supported Best-Effort Forms

Tier 2 forms are supported, maintained, and recommended where their scope is useful, but they are not part of the strongest contractual surface.

They may be:

- lossy by design;
- partially supported;
- stable in daily use without carrying full Tier 1 parity guarantees;
- likely candidates for promotion once their scope and verification harden further.

### Tier 3 — Experimental, Transitional, or Niche Forms

Tier 3 forms are available but outside the contractual core of the `1.0.0` line.

They may be:

- experimental;
- specialized;
- legacy;
- transitional;
- or insufficiently verified for contractual support.

Tier 3 forms are useful to retain, but they should not be presented as part of the guaranteed production-grade core.
> **Authority:** `molsysmt/_private/form_tier.py` is the runtime source of truth. This notebook is an executable report and must not carry a separate hand-maintained classification table.


In [ ]:
# Programmatic tier registry — single source of truth for runtime signals and pytest marks.
# Every discovered form, including Tier 1, must be present in FORM_TIERS.

from molsysmt._private.form_tier import FORM_TIERS
import molsysmt.form as _msm_form_pkg
import re
from pathlib import Path

# Discover all form directory names and their form_name
form_base = Path(_msm_form_pkg.__file__).parent
pattern = re.compile(r"^form_name\s*=\s*['\"]([^'\"]+)['\"]", re.MULTILINE)

all_forms = {}
for form_dir in sorted(form_base.iterdir()):
    if not form_dir.is_dir() or form_dir.name.startswith('_'):
        continue
    init_file = form_dir / '__init__.py'
    if not init_file.exists():
        continue
    text = init_file.read_text()
    m = pattern.search(text)
    if m:
        form_name = m.group(1)
        tier = FORM_TIERS[form_name]
        all_forms[form_name] = tier

# Display by tier
for tier_n in [1, 2, 3]:
    forms = sorted(k for k, v in all_forms.items() if v == tier_n)
    print(f"--- Tier {tier_n} ({len(forms)} forms) ---")
    for f in forms:
        print(f"  {f}")
    print()

In [ ]:
# Sanity check: the registry and discovered adapters must match exactly.
known_dirs = set(all_forms.keys())
missing = sorted(known_dirs - set(FORM_TIERS))
stale = sorted(set(FORM_TIERS) - known_dirs)
if missing or stale:
    raise RuntimeError(f"Form-tier registry mismatch: missing={missing}, stale={stale}")
else:
    print("OK — every discovered form has exactly one explicit tier entry.")

> **Generated view:** Run the two cells above to inspect the current registry. Contract, parity, and heavy-mode claims belong in executable test reports or the relevant normative guide; they are deliberately not duplicated here.
